# Module 06 Lab — CrewAI: Hierarchical Research Crew

**Course:** AI Agents Teaching Platform  
**Module:** 06 — Multi-Agent Systems  
**Estimated time:** ~90 minutes

---

### What you'll build

A hierarchical research crew using `Process.hierarchical`:
- **Orchestrator** (manager_agent) — allocates sub-tasks and synthesises
- **Senior Researcher** — finds and evaluates technical evidence
- **Data Analyst** — quantifies findings with numbers and metrics
- **Report Writer** — structures the final written report

Then: measure token overhead of hierarchical vs sequential coordination.

> **Note:** Uses `langchain-anthropic` as the CrewAI LLM backend.

In [ ]:
%pip install -q crewai crewai-tools langchain-anthropic

In [ ]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Paste your Anthropic API key: ")
print("Key set ✓")

## Step 1 — Configure the LLM

CrewAI uses LangChain LLM objects. We use `claude-haiku-4-5-20251001` for all agents
to keep costs manageable — you can switch workers to a cheaper model independently.

In [ ]:
from langchain_anthropic import ChatAnthropic
from crewai import Agent, Task, Crew, Process

llm = ChatAnthropic(model="claude-haiku-4-5-20251001", max_tokens=512)
print("LLM configured ✓")

## Step 2 — Define the agents (TODO)

Each CrewAI agent has three key fields:
- `role` — short job title (used in inter-agent routing)
- `goal` — one-sentence objective
- `backstory` — 2–3 sentences giving expertise context

The Orchestrator and Senior Researcher are complete.
**TODO:** Fill in the `goal` and `backstory` for the Data Analyst and Report Writer.

In [ ]:
orchestrator = Agent(
    role="Research Orchestrator",
    goal="Decompose complex research questions and synthesise findings from specialist agents into a coherent, evidence-based answer.",
    backstory="""You are an experienced research director who has led multidisciplinary teams for 15 years.
You know how to break a big question into tractable sub-problems, delegate to specialists,
and synthesise their outputs without losing nuance.""",
    llm=llm, verbose=True, allow_delegation=True,
)

senior_researcher = Agent(
    role="Senior Researcher",
    goal="Find and critically evaluate the strongest technical evidence on a given topic, highlighting both consensus and controversy.",
    backstory="""With a PhD in computer science and a decade of research experience, you specialise in
AI systems and machine learning. You know which papers matter, which results replicate,
and how to distinguish hype from substance.""",
    llm=llm, verbose=True, allow_delegation=False,
)

data_analyst = Agent(
    role="Data Analyst",
    goal="TODO: one sentence describing what this agent optimises for.",
    backstory="""TODO: 2-3 sentences of expertise context that makes the agent's quantitative focus credible.""",
    llm=llm, verbose=True, allow_delegation=False,
)

report_writer = Agent(
    role="Report Writer",
    goal="TODO: one sentence describing what this agent produces.",
    backstory="""TODO: 2-3 sentences that establish credibility in technical communication.""",
    llm=llm, verbose=True, allow_delegation=False,
)

print("Agents defined ✓")

<details>
<summary>💡 Stuck? Reveal agent field solutions</summary>

```python
# Data Analyst
goal="Extract, verify, and present quantitative evidence — numbers, benchmarks, and metrics — from research findings."
backstory="""You specialise in quantitative research methods and have worked as a data scientist
for leading AI labs. You are skilled at turning qualitative claims into measurable assertions
and identifying when numbers are missing or misleading."""

# Report Writer
goal="Transform raw findings and data into a clearly structured, reader-friendly research report with executive summary and limitations."
backstory="""You are a senior technical writer with experience at top-tier journals and research institutions.
You know how to distil dense material into compelling prose without losing precision,
and you structure every report for a non-specialist executive audience."""
```

</details>

## Step 3 — Define the tasks (TODO)

Each `Task` has a `description`, an `expected_output`, and an `agent`.
The `research_task` is complete. **TODO:** Fill in `analysis_task` and `report_task`.

In [ ]:
RESEARCH_QUESTION = "What are the main technical bottlenecks preventing wider deployment of large language models?"

research_task = Task(
    description=f"""Research the following question and identify the 3–4 most important technical findings:
{RESEARCH_QUESTION}
Focus on inference cost, memory requirements, latency, and alignment challenges.
For each finding, explain the mechanism and cite any known solutions or workarounds.""",
    expected_output="A structured list of 3–4 technical findings, each with a 2–3 sentence explanation.",
    agent=senior_researcher,
)

analysis_task = Task(
    description="""TODO: Describe what the Data Analyst should do with the research findings.
Include: what quantitative evidence to extract, what to benchmark, what gaps to flag.""",
    expected_output="TODO: describe the expected output format (numbers, tables, metrics).",
    agent=data_analyst,
    context=[research_task],
)

report_task = Task(
    description="""TODO: Describe what the Report Writer should produce given all prior outputs.
Specify: structure (headings), length, required sections (summary, findings, limitations).""",
    expected_output="TODO: describe the final report format.",
    agent=report_writer,
    context=[research_task, analysis_task],
)

print("Tasks defined ✓")

<details>
<summary>💡 Stuck? Reveal task solutions</summary>

```python
analysis_task = Task(
    description="""Based on the research findings, extract and verify all quantitative claims:
- Memory requirements (GB for typical model sizes)
- Latency benchmarks (tokens/second for various hardware)
- Cost estimates ($/1k tokens on major providers)
Where numbers are missing, flag them explicitly as 'unquantified'.""",
    expected_output="A table or structured list of quantitative metrics with sources/confidence notes.",
    agent=data_analyst,
    context=[research_task],
)

report_task = Task(
    description="""Synthesise the research findings and quantitative analysis into a final report.
Structure:
## Executive Summary (2 sentences)
## Key Technical Bottlenecks (one subsection per bottleneck)
## Quantitative Overview (the metrics table from the analyst)
## Limitations and Open Questions
Total length: 400–500 words.""",
    expected_output="A polished markdown report with all four sections.",
    agent=report_writer,
    context=[research_task, analysis_task],
)
```

</details>

## Step 4 — Build the hierarchical crew and run it

In [ ]:
crew = Crew(
    agents=[senior_researcher, data_analyst, report_writer],
    tasks=[research_task, analysis_task, report_task],
    process=Process.hierarchical,
    manager_agent=orchestrator,
    verbose=True,
)

result = crew.kickoff()
print("\n--- Final Report ---")
print(result)

## Step 5 — Compare to sequential (TODO)

**TODO:** Create a second crew with the same agents and tasks but `Process.sequential`.
Run it and compare the output and token count to the hierarchical version.

In [ ]:
# TODO: create sequential_crew = Crew(..., process=Process.sequential)
# Note: sequential process does NOT use manager_agent
# sequential_result = sequential_crew.kickoff()

# After both runs, compare:
# print("Hierarchical token cost:", crew.usage_metrics)
# print("Sequential token cost:", sequential_crew.usage_metrics)

print("Fill in sequential_crew above")

<details>
<summary>💡 Stuck? Reveal the sequential crew solution</summary>

```python
sequential_crew = Crew(
    agents=[senior_researcher, data_analyst, report_writer],
    tasks=[research_task, analysis_task, report_task],
    process=Process.sequential,
    verbose=True,
)

sequential_result = sequential_crew.kickoff()
print("Hierarchical usage:", crew.usage_metrics)
print("Sequential usage:",   sequential_crew.usage_metrics)
```

Expect hierarchical to use ~30–50% more tokens due to manager overhead.
The trade-off: hierarchical handles mid-task replanning; sequential cannot.

</details>

## Step 6 — Exercises

### Exercise 1: Role specialisation
Add a fourth worker agent with `role="Devil's Advocate"` whose goal is to argue
against the consensus. Add a corresponding task. Does the final report change?

### Exercise 2: allow_delegation
Change `data_analyst`'s `allow_delegation=True`. Does the manager start routing
sub-tasks to the analyst directly? How does this affect total rounds?

### Exercise 3: Tool use
Import `from crewai_tools import SerperDevTool` (requires `SERPER_API_KEY`).
Attach it to the researcher: `tools=[SerperDevTool()]`. How does grounded search change the output?

### Exercise 4: Switch models
Change `llm = ChatAnthropic(model="claude-sonnet-4-6")` for the orchestrator only.
Does giving the manager a smarter model improve task allocation?

In [ ]:
# Scratch cell — use for exercises


## Step 7 — Reflection

1. What does `context=[research_task]` do in the analysis task? How does it differ from passing data via function arguments?
2. The hierarchical process uses more tokens than sequential. When is that overhead worth it?
3. What happens if `manager_agent`'s `allow_delegation=True` is False? (Hint: try it.)
4. How does CrewAI's `role/goal/backstory` format compare to LangGraph's `TypedDict` state for maintaining shared context?

*(Double-click to edit)*

1. 
2. 
3. 
4. 